#### 1. 没有 \Psi 语义投影空间的双阈值代理过滤AQP 
下面是运行命令
python Cascade-Filter.py \
  --parent_dataset amazon_data \
  --dataset amazon_extend \
  --fastest_bin /home/wangshuo/projects/FaSTest-main/build/Fastest \
  --ablation_csv /home/wangshuo/resource/datasets/amazon_data/amazon_extend/results/efficiency/allocation_strategy_comparison_ablation_sum.csv \
  --table1 product \
  --table1_proxy ML3_proxy2_probability \
  --table1_oracle ML3_oracle2_probability \
  --table2 review \
  --table2_proxy ML2_proxy2_probability \
  --table2_oracle ML2_oracle1_probability \
  --sum_col price \
  --sum_label 12 \
  --t1_low 0.2 --t1_high 0.3 \
  --t2_low 0.2 --t2_high 0.3 \
  --num_workers 8

python Cascade-Filter.py \
  --parent_dataset parler \
  --dataset parler \
  --fastest_bin /home/wangshuo/projects/FaSTest-main/build/Fastest \
  --ablation_csv /home/wangshuo/resource/datasets/amazon_data/amazon_extend/results/efficiency/allocation_strategy_comparison_ablation_sum.csv \
  --table1 product \
  --table1_proxy ML3_proxy2_probability \
  --table1_oracle ML3_oracle2_probability \
  --table2 review \
  --table2_proxy ML2_proxy2_probability \
  --table2_oracle ML2_oracle1_probability \
  --sum_col price \
  --sum_label 12 \
  --t1_low 0.2 --t1_high 0.3 \
  --t2_low 0.2 --t2_high 0.3 \
  --num_workers 8

In [2]:
import json
import numpy as np
import pandas as pd

# ==========================================
# 1. 文件路径配置（请按需修改路径）
# ==========================================
# 替换为你的 AQP_Cacade_results.csv 的实际路径
csv_file = "/home/wangshuo/resource/datasets/amazon_data/amazon_extend/results/efficiency/AQP_Cascade_results.csv"  
gt_file = "/home/wangshuo/resource/datasets/amazon_data/amazon_extend/results/T_true_ML3_oracle2_probability_ML2_oracle1_probability_sum.json"

# ==========================================
# 2. 读取数据
# ==========================================
# 2.1 读取 CSV 估计数据
df_est = pd.read_csv(csv_file)

# 2.2 读取 JSON Ground Truth 数据
with open(gt_file, "r", encoding="utf-8") as f:
    gt_data = json.load(f)

# ==========================================
# 3. 数据对齐与误差计算
# ==========================================
records = []
epsilon = 1e-9  # 防除零保护

for _, row in df_est.iterrows():
    raw_key = str(row["query_basename"]).strip()
    t_hat = float(row["T_hat_aqp"])
    
    # 统一清洗 key（去除可能存在的 .graph 后缀，防止对齐失败）
    clean_key = raw_key.replace(".graph", "")
    
    # 寻找 ground truth 中对应的真实值（尝试完整 key、带 .graph 后缀和不带后缀三种情况）
    t_true = gt_data.get(raw_key)
    if t_true is None:
        t_true = gt_data.get(f"{clean_key}.graph")
    if t_true is None:
        t_true = gt_data.get(clean_key)
        
    # 过滤无效值（仅对比两边都有且 T_true > 0 的查询）
    if t_true is not None and t_true > 0:
        # 带符号相对误差 (Signed Relative Error): (估计值 - 真实值) / 真实值
        signed_re = (t_hat - t_true) / (t_true + epsilon)
        
        # 绝对值相对误差 (Absolute Relative Error, ARE): |估计值 - 真实值| / 真实值
        are = abs(t_hat - t_true) / (t_true + epsilon)
        
        records.append({
            "query": clean_key,
            "T_hat_aqp (估计值)": t_hat,
            "T_true (真实值)": t_true,
            "Signed_RE (带符号误差)": signed_re,
            "ARE (绝对误差)": are
        })

df = pd.DataFrame(records)

# ==========================================
# 4. 汇总统计与输出
# ==========================================
if df.empty:
    print("❌ 未成功匹配到任何查询，请检查 CSV 中的 query_basename 与 JSON 文件中的 key 名称是否一致！")
else:
    mean_signed_re = df["Signed_RE (带符号误差)"].mean()
    mean_are = df["ARE (绝对误差)"].mean()
    
    # 各种分位数计算 (P50/中位数, P70, P90, P95)
    p50_are = df["ARE (绝对误差)"].median()  # 等价于 .quantile(0.50)
    p70_are = df["ARE (绝对误差)"].quantile(0.70)
    p90_are = df["ARE (绝对误差)"].quantile(0.90)
    p95_are = df["ARE (绝对误差)"].quantile(0.95)

    print("=" * 65)
    print(f"📊 AQP Cascade 代理方法评估结果汇总 (成功对齐 {len(df)} 个查询)")
    print("=" * 65)
    print(f"1. 带符号相对误差均值 (Mean Signed RE) : {mean_signed_re:.4f}  ({mean_signed_re * 100:.2f}%)")
    print(f"2. 绝对值相对误差均值 (Mean ARE)       : {mean_are:.4f}  ({mean_are * 100:.2f}%)")
    print("-" * 65)
    print(f"3. 绝对值相对误差中位数 (P50 ARE)       : {p50_are:.4f}  ({p50_are * 100:.2f}%)")
    print(f"4. 绝对值相对误差 P70 (P70 ARE)         : {p70_are:.4f}  ({p70_are * 100:.2f}%)")
    print(f"5. 绝对值相对误差 P90 (P90 ARE)         : {p90_are:.4f}  ({p90_are * 100:.2f}%)")
    print(f"6. 绝对值相对误差 P95 (P95 ARE)         : {p95_are:.4f}  ({p95_are * 100:.2f}%)")
    print("=" * 65)
    
    # 格式化百分比显示详细表格
    df_display = df.copy()
    df_display["Signed_RE (带符号误差)"] = df_display["Signed_RE (带符号误差)"].map("{:.2%}".format)
    df_display["ARE (绝对误差)"] = df_display["ARE (绝对误差)"].map("{:.2%}".format)
    
    # 兼容 Jupyter display 和 终端 print
    try:
        display(df_display.head(10))
    except NameError:
        print(df_display.head(10).to_string(index=False))

📊 AQP Cascade 代理方法评估结果汇总 (成功对齐 176 个查询)
1. 带符号相对误差均值 (Mean Signed RE) : -0.2710  (-27.10%)
2. 绝对值相对误差均值 (Mean ARE)       : 0.4123  (41.23%)
-----------------------------------------------------------------
3. 绝对值相对误差中位数 (P50 ARE)       : 0.3100  (31.00%)
4. 绝对值相对误差 P70 (P70 ARE)         : 0.5738  (57.38%)
5. 绝对值相对误差 P90 (P90 ARE)         : 0.8839  (88.39%)
6. 绝对值相对误差 P95 (P95 ARE)         : 0.9389  (93.89%)


,query,T_hat_aqp (估计值),T_true (真实值),Signed_RE (带符号误差),ARE (绝对误差)
0,query_5_12,559.62,1202.53,-53.46%,53.46%
1,query_5_117,795.86,754.83,5.44%,5.44%
2,query_5_101,330.44,1147.20,-71.20%,71.20%
3,query_5_113,320.64,225.31,42.31%,42.31%
4,query_5_2,268.58,792.72,-66.12%,66.12%
5,query_5_11,158.49,657.40,-75.89%,75.89%
6,query_5_116,691.68,1180.76,-41.42%,41.42%
7,query_5_120,634.69,503.95,25.94%,25.94%
8,query_5_33,161.87,1291.29,-87.46%,87.46%
9,query_5_37,253.67,866.33,-70.72%,70.72%


#### 2. 基于 \PSi 的双阈值截断, 双阈值根据GT选择F1分数最高的 Proxy 阈值, 效果肯定 >= SUPG/ScaleDoc的阈值, 然后同时引入灰色区间Oracle介入,进一步提高准确性
上面1,2基线的作用是突出我\Psi 语义投影空间的作用, 以及我方法的无偏性和低方差.

In [9]:
import json
import os
import numpy as np
import pandas as pd

# ==========================================
# 1. 路径配置（根据您的实际文件名修改 csv_name）
# ==========================================
dataset_dir = "/home/wangshuo/resource/datasets/amazon_data/amazon_extend/results"

# AQP 结果 CSV 路径 (如 AQP_Cascade_results.csv 或 Core_Double_Truncation_amazon_sum.csv)
csv_path = os.path.join(dataset_dir, "efficiency", "Core_Double_Truncation_amazon_sum.csv")

# Ground Truth JSON 路径
gt_json_path = os.path.join(dataset_dir, "T_true_ML3_oracle2_probability_ML2_oracle1_probability_sum.json")

# ==========================================
# 2. 数据读取与自动对齐
# ==========================================
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"❌ 找不到估计结果 CSV 文件: {csv_path}")

if not os.path.exists(gt_json_path):
    raise FileNotFoundError(f"❌ 找不到 Ground Truth JSON 文件: {gt_json_path}")

# 读取 Ground Truth 并归一化键名
with open(gt_json_path, 'r', encoding='utf-8') as f:
    gt_dict = json.load(f)
gt_map = {str(k).replace(".graph", ""): float(v) for k, v in gt_dict.items() if v is not None}

# 读取 AQP 估计结果 CSV
df = pd.read_csv(csv_path)

# 自动识别估计值列名
est_col = None
for col in ["T_hat_aqp", "T_hat_double_truncation", "T_hat", "estimate"]:
    if col in df.columns:
        est_col = col
        break

if not est_col:
    raise ValueError(f"❌ CSV 中未找到估计值列！当前列名: {df.columns.tolist()}")

# ==========================================
# 3. 逐查询匹配与误差计算
# ==========================================
records = []
epsilon = 1e-9

for _, row in df.iterrows():
    q_name = str(row["query_basename"]).replace(".graph", "").replace(".csv", "")
    t_hat = float(row[est_col])
    
    t_true = gt_map.get(q_name)
    if t_true is not None and t_true > 0:
        # 带符号相对误差 (Signed RE): (估计值 - 真实值) / 真实值
        signed_re = (t_hat - t_true) / (t_true + epsilon)
        
        # 绝对值相对误差 (ARE): |估计值 - 真实值| / 真实值
        are = abs(t_hat - t_true) / (t_true + epsilon)
        
        records.append({
            "query": q_name,
            "T_hat": t_hat,
            "T_true": t_true,
            "Signed_RE": signed_re,
            "ARE": are
        })

eval_df = pd.DataFrame(records)

# ==========================================
# 4. 指标统计与汇总框输出
# ==========================================
if eval_df.empty:
    print("❌ 未成功对齐到任何有效查询记录，请检查 CSV 与 GT JSON 的 key 名称！")
else:
    mean_signed_re = eval_df["Signed_RE"].mean()
    mean_are = eval_df["ARE"].mean()
    p50_are = eval_df["ARE"].median()
    p70_are = eval_df["ARE"].quantile(0.70)
    p90_are = eval_df["ARE"].quantile(0.90)
    p95_are = eval_df["ARE"].quantile(0.95)

    print("=" * 65)
    print(f"📊 AQP Cascade 代理方法评估结果汇总 (成功对齐 {len(eval_df)} 个查询)")
    print("=" * 65)
    print(f"1. 带符号相对误差均值 (Mean Signed RE) : {mean_signed_re:.4f}  ({mean_signed_re * 100:.2f}%)")
    print(f"2. 绝对值相对误差均值 (Mean ARE)       : {mean_are:.4f}  ({mean_are * 100:.2f}%)")
    print("-" * 65)
    print(f"3. 绝对值相对误差中位数 (P50 ARE)       : {p50_are:.4f}  ({p50_are * 100:.2f}%)")
    print(f"4. 绝对值相对误差 P70 (P70 ARE)         : {p70_are:.4f}  ({p70_are * 100:.2f}%)")
    print(f"5. 绝对值相对误差 P90 (P90 ARE)         : {p90_are:.4f}  ({p90_are * 100:.2f}%)")
    print(f"6. 绝对值相对误差 P95 (P95 ARE)         : {p95_are:.4f}  ({p95_are * 100:.2f}%)")
    print("=" * 65)

📊 AQP Cascade 代理方法评估结果汇总 (成功对齐 180 个查询)
1. 带符号相对误差均值 (Mean Signed RE) : 0.0264  (2.64%)
2. 绝对值相对误差均值 (Mean ARE)       : 0.1247  (12.47%)
-----------------------------------------------------------------
3. 绝对值相对误差中位数 (P50 ARE)       : 0.0747  (7.47%)
4. 绝对值相对误差 P70 (P70 ARE)         : 0.1122  (11.22%)
5. 绝对值相对误差 P90 (P90 ARE)         : 0.2411  (24.11%)
6. 绝对值相对误差 P95 (P95 ARE)         : 0.3295  (32.95%)


#### 3.  abae论文也用到我的语义投影集上, projection-abaze, 就先将每个核心实例按照代理分数排序, 通过 pilot 和  简单分层采样(层内是均匀采样)  估计avg :这个基线的作用是突出我重要性采样的作用以及无pilot,闭式分配的作用

python PRO-ABAE.py   --parent_dataset amazon_data   --dataset_name amazon_extend   --ablation_csv /home/wangshuo/resource/datasets/amazon_data/amazon_extend/results/efficiency/allocation_strategy_comparison_ablation_sum.csv   --t1_proxy ML3_proxy2_probability   --t1_oracle ML3_oracle2_probability   --t2_proxy ML2_proxy2_probability   --t2_oracle ML2_oracle1_probability   --workers 16   --out_csv Projection_ABae_amazon_sum.csv

python PROJ-ABAE.py   --parent_dataset parler_data   --dataset_name dataset_three   --ablation_csv /home/wangshuo/resource/datasets/parler_data/dataset_three/results/efficiency/allocation_strategy_comparison_ablation_count.csv   --t1_proxy ML1_proxy4b_probability   --t1_oracle ML1_oracle2_probability   --t2_proxy ML2_proxy1_probability   --t2_oracle ML2_oracle2_probability   --workers 16   --out_csv Projection_ABae_amazon_count.csv

In [13]:
import json
import os
import numpy as np
import pandas as pd

# ==========================================
# 1. 路径与目标配置 (Parler-E 数据集)
# ==========================================
parent_dataset = "parler_data"
dataset_name = "dataset_three"

base_results_dir = f"/home/wangshuo/resource/datasets/{parent_dataset}/{dataset_name}/results"

# 估计结果 CSV 路径
csv_path = os.path.join(base_results_dir, "efficiency", "Projection_ABae_amazon_count.csv")
# 兜底路径（如果上面找不到，尝试默认文件名）
if not os.path.exists(csv_path):
    csv_path = os.path.join(base_results_dir, "efficiency", "Projection_ABae_results_count.csv")

# Ground Truth JSON 路径 (Parler SUM)
# gt_json_path = os.path.join(base_results_dir, "T_true_ML3_oracle2_probability_ML2_oracle1_probability_sum.json")
_json_path = os.path.join(base_results_dir, "T_true_ML1_oracle2_probability_ML2_oracle2_probability_count.json")

# ==========================================
# 2. 读取与预处理
# ==========================================
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"❌ 找不到估计结果 CSV 文件: {csv_path}")

if not os.path.exists(gt_json_path):
    raise FileNotFoundError(f"❌ 找不到 Ground Truth JSON 文件: {gt_json_path}")

# 读取真值 JSON 并归一化键名 (去 .graph)
with open(gt_json_path, 'r', encoding='utf-8') as f:
    gt_dict = json.load(f)
gt_map = {str(k).replace(".graph", ""): float(v) for k, v in gt_dict.items() if v is not None and v > 0}

# 读取 Projection-ABae 结果 CSV
df = pd.read_csv(csv_path)

# 自动匹配估计值列名
est_col = None
for col in ["T_hat_abae", "T_hat_abae_avg", "T_hat", "T_hat_sum"]:
    if col in df.columns:
        est_col = col
        break

if not est_col:
    raise ValueError(f"❌ CSV 中未找到估计值列！当前列名: {df.columns.tolist()}")

# 3) 规范化 query_basename 键名
df["query_clean"] = df["query_basename"].astype(str).str.replace(r"\.graph$", "", regex=True)

# 如果存在多轮 run_id，按 query 对估计值 T_hat 求均值
if "run_id" in df.columns:
    df_eval = df.groupby("query_clean")[est_col].mean().reset_index()
else:
    df_eval = df[["query_clean", est_col]].copy()

# ==========================================
# 3. 对齐真值与计算误差
# ==========================================
records = []
epsilon = 1e-9

for _, row in df_eval.iterrows():
    q_name = row["query_clean"]
    t_hat = float(row[est_col])
    
    t_true = gt_map.get(q_name)
    if t_true is not None and t_true > 0:
        # 带符号相对误差 (Signed RE): (估计值 - 真实值) / 真实值
        signed_re = (t_hat - t_true) / (t_true + epsilon)
        
        # 绝对值相对误差 (ARE): |估计值 - 真实值| / 真实值
        are = abs(t_hat - t_true) / (t_true + epsilon)
        
        records.append({
            "query": q_name,
            "T_hat": t_hat,
            "T_true": t_true,
            "Signed_RE": signed_re,
            "ARE": are
        })

eval_df = pd.DataFrame(records)

# ==========================================
# 4. 指标统计与汇总框输出
# ==========================================
if eval_df.empty:
    print("❌ 未成功对齐到任何有效查询记录，请检查 CSV 与 GT JSON 的 key 名称！")
else:
    mean_signed_re = eval_df["Signed_RE"].mean()
    mean_are = eval_df["ARE"].mean()
    p50_are = eval_df["ARE"].median()
    p70_are = eval_df["ARE"].quantile(0.70)
    p90_are = eval_df["ARE"].quantile(0.90)
    p95_are = eval_df["ARE"].quantile(0.95)

    print("=" * 65)
    print(f"📊 Projection-ABae 方法评估结果汇总 (成功对齐 {len(eval_df)} 个查询)")
    print("=" * 65)
    print(f"1. 带符号相对误差均值 (Mean Signed RE) : {mean_signed_re:.4f}  ({mean_signed_re * 100:.2f}%)")
    print(f"2. 绝对值相对误差均值 (Mean ARE)       : {mean_are:.4f}  ({mean_are * 100:.2f}%)")
    print("-" * 65)
    print(f"3. 绝对值相对误差中位数 (P50 ARE)       : {p50_are:.4f}  ({p50_are * 100:.2f}%)")
    print(f"4. 绝对值相对误差 P70 (P70 ARE)         : {p70_are:.4f}  ({p70_are * 100:.2f}%)")
    print(f"5. 绝对值相对误差 P90 (P90 ARE)         : {p90_are:.4f}  ({p90_are * 100:.2f}%)")
    print(f"6. 绝对值相对误差 P95 (P95 ARE)         : {p95_are:.4f}  ({p95_are * 100:.2f}%)")
    print("=" * 65)



📊 Projection-ABae 方法评估结果汇总 (成功对齐 246 个查询)
1. 带符号相对误差均值 (Mean Signed RE) : -0.0227  (-2.27%)
2. 绝对值相对误差均值 (Mean ARE)       : 0.2363  (23.63%)
-----------------------------------------------------------------
3. 绝对值相对误差中位数 (P50 ARE)       : 0.1918  (19.18%)
4. 绝对值相对误差 P70 (P70 ARE)         : 0.3069  (30.69%)
5. 绝对值相对误差 P90 (P90 ARE)         : 0.4773  (47.73%)
6. 绝对值相对误差 P95 (P95 ARE)         : 0.5608  (56.08%)


#### 4. Our Proxy 算法的输出结果

In [2]:
import json
import os
import numpy as np
import pandas as pd

# ==========================================
# 1. 路径与目标配置
# ==========================================

dataset_dir = "../../../datasets/parler-E/results"
# 优先读取消融 CSV 文件，若不存在则读取常规 CSV
csv_path = os.path.join(dataset_dir, "efficiency", "allocation_strategy_comparison_count.csv")
if not os.path.exists(csv_path):
    csv_path = os.path.join(dataset_dir, "efficiency", "allocation_strategy_comparison_count.csv")

# Ground Truth JSON 路径
# gt_json_path = os.path.join(dataset_dir, "T_true_ML3_oracle2_probability_ML2_oracle1_probability_sum.json")
gt_json_path = os.path.join(dataset_dir, "T_true_ML1_oracle2_probability_ML2_oracle2_probability_count.json")

TARGET_METHOD = "8_POSSA"  # 目标方法（兼容 8_POSSA 或 POSS）
TARGET_FRAC = 0.1          # 目标预算比例

# ==========================================
# 2. 数据读取与预处理
# ==========================================
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"❌ 找不到 CSV 文件: {csv_path}")

if not os.path.exists(gt_json_path):
    raise FileNotFoundError(f"❌ 找不到 Ground Truth JSON 文件: {gt_json_path}")

# 读取 Ground Truth JSON 并归一化键名
with open(gt_json_path, 'r', encoding='utf-8') as f:
    gt_dict = json.load(f)
gt_map = {str(k).replace(".graph", ""): float(v) for k, v in gt_dict.items() if v is not None}

# 读取 CSV 文件
df = pd.read_csv(csv_path)

# 1) 过滤方法 (8_POSSA 或 POSS)
df_filtered = df[df["method"].isin(["8_POSSA", "POSS"])].copy()

# 2) 过滤采样率 (budget_frac == 0.1)
if "budget_frac" in df_filtered.columns:
    df_filtered = df_filtered[np.isclose(df_filtered["budget_frac"], TARGET_FRAC, atol=1e-4)].copy()

if df_filtered.empty:
    raise ValueError(f"❌ 未找到符合 method={TARGET_METHOD} 且 budget_frac={TARGET_FRAC} 的记录！")

# 3) 规范化 query_basename 键名
df_filtered["query_clean"] = df_filtered["query_basename"].astype(str).str.replace(r"\.graph$", "", regex=True)

# ==========================================
# 3. 对齐真值与计算误差 (完全按脚本 1 粒度：每条运行记录独立计算)
# ==========================================
records = []
epsilon = 1e-9

# 直接遍历每一条运行记录（不提前 groupby 求 T_hat 平均）
for _, row in df_filtered.iterrows():
    q_name = row["query_clean"]
    t_hat = float(row["T_hat"])
    
    t_true = gt_map.get(q_name)
    if t_true is not None and t_true > 0:
        # 带符号相对误差 (Signed RE): (估计值 - 真实值) / 真实值
        signed_re = (t_hat - t_true) / (t_true + epsilon)
        
        # 绝对值相对误差 (ARE): |估计值 - 真实值| / 真实值
        are = abs(t_hat - t_true) / (t_true + epsilon)
        
        records.append({
            "query": q_name,
            "run_id": row.get("run_id", 1),
            "T_hat": t_hat,
            "T_true": t_true,
            "Signed_RE": signed_re,
            "ARE": are
        })

eval_df = pd.DataFrame(records)

# ==========================================
# 4. 指标统计与汇总框输出
# ==========================================
if eval_df.empty:
    print("❌ 未成功对齐到任何有效查询记录，请检查 CSV 与 GT JSON 的 key 名称！")
else:
    mean_signed_re = eval_df["Signed_RE"].mean()
    mean_are = eval_df["ARE"].mean()
    p50_are = eval_df["ARE"].median()
    p70_are = eval_df["ARE"].quantile(0.70)
    p90_are = eval_df["ARE"].quantile(0.90)
    p95_are = eval_df["ARE"].quantile(0.95)

    print("=" * 65)
    print(f"📊 8_POSSA (POSS) 方法评估结果汇总 (完全对齐脚本 1 粒度)")
    print(f"   数据文件: {os.path.basename(csv_path)} | 匹配记录数: {len(eval_df)} 条")
    print("=" * 65)
    print(f"1. 带符号相对误差均值 (Mean Signed RE) : {mean_signed_re:.4f}  ({mean_signed_re * 100:.2f}%)")
    print(f"2. 绝对值相对误差均值 (Mean ARE)       : {mean_are:.4f}  ({mean_are * 100:.2f}%)")
    print("-" * 65)
    print(f"3. 绝对值相对误差中位数 (P50 ARE)       : {p50_are:.4f}  ({p50_are * 100:.2f}%)")
    print(f"4. 绝对值相对误差 P70 (P70 ARE)         : {p70_are:.4f}  ({p70_are * 100:.2f}%)")
    print(f"5. 绝对值相对误差 P90 (P90 ARE)         : {p90_are:.4f}  ({p90_are * 100:.2f}%)")
    print(f"6. 绝对值相对误差 P95 (P95 ARE)         : {p95_are:.4f}  ({p95_are * 100:.2f}%)")
    print("=" * 65)

📊 8_POSSA (POSS) 方法评估结果汇总 (完全对齐脚本 1 粒度)
   数据文件: allocation_strategy_comparison_count.csv | 匹配记录数: 550 条
1. 带符号相对误差均值 (Mean Signed RE) : -0.0108  (-1.08%)
2. 绝对值相对误差均值 (Mean ARE)       : 0.0516  (5.16%)
-----------------------------------------------------------------
3. 绝对值相对误差中位数 (P50 ARE)       : 0.0257  (2.57%)
4. 绝对值相对误差 P70 (P70 ARE)         : 0.0451  (4.51%)
5. 绝对值相对误差 P90 (P90 ARE)         : 0.1103  (11.03%)
6. 绝对值相对误差 P95 (P95 ARE)         : 0.1623  (16.23%)
